In [23]:
!uv add \
  langchain \
  langchain-community \
  langchain-huggingface \
  langchain_text_splitters \
  langchain_core \
  langsmith chromadb \
  sentence-transformers \
  transformers \
  torch \
  huggingface_hub \
  python-dotenv \
  datasets \
  ragas \
  tqdm

Resolved 163 packages in 10ms
Checked 140 packages in 23ms


In [24]:
import warnings
warnings.filterwarnings('ignore')

In [25]:
from dotenv import load_dotenv
load_dotenv()

import os

LANGCHAIN_TRACING_V2 = os.getenv("LANGCHAIN_TRACING_V2")
LANGCHAIN_ENDPOINT = os.getenv("LANGCHAIN_ENDPOINT")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT = os.getenv("LANGCHAIN_PROJECT")
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [26]:
ARTICLES_PATH = "./data/articles.json"
CHROMA_DB_PATH = "./data/chromadb"
COLLECTION_NAME = "newspaper_archive"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "google/flan-t5-large"

In [27]:
import json
import pandas as pd

# load file
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    raw_articles = json.load(f)

# file preview as dataframe
df = pd.DataFrame(raw_articles)
df.head()

,id,title,date,author,section,content
0,ART-001,City Council Approves ₹500 Crore Infrastructur...,2023-03-15,Priya Menon,City,The Chennai City Council unanimously approved ...
1,ART-002,Local Startup EcoRide Raises ₹40 Crore in Seri...,2023-06-02,Karthik Sundaram,Business,Chennai-based electric vehicle startup EcoRide...
2,ART-003,Marina Beach Plastic Drive Collects 8 Tonnes o...,2022-09-25,Lakshmi Iyer,Environment,A three-day beach cleanup drive at Marina Beac...
3,ART-004,Government School Achieves 100% Class X Pass Rate,2023-05-10,Meena Krishnaswamy,Education,Rajaji Government Higher Secondary School in R...
4,ART-005,"Flooding in Tambaram Displaces 1,200 Families",2023-11-18,Arjun Balaji,Disaster,Heavy overnight rainfall of 21 cm caused sever...


In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks = []

# individual articles
for article in raw_articles:
    full_text = article["content"]
    text_chunks = splitter.split_text(full_text)

    for i, chunk in enumerate(text_chunks):
        doc = Document(
            page_content=chunk,
            metadata={
                "article_id": article["id"],
                "title": article["title"],
                "date": article["date"],
                "author": article["author"],
                "section": article["section"],
                "chunk_index": i,
                "total_chunks": len(text_chunks)
            }
        )
        all_chunks.append(doc)

all_chunks[:5]

[Document(metadata={'article_id': 'ART-001', 'title': 'City Council Approves ₹500 Crore Infrastructure Plan', 'date': '2023-03-15', 'author': 'Priya Menon', 'section': 'City', 'chunk_index': 0, 'total_chunks': 3}, page_content="The Chennai City Council unanimously approved a ₹500 crore infrastructure development plan on Tuesday, aimed at upgrading roads, drainage systems, and public parks across 15 wards. Mayor Suresh Kumar called it the largest single-year civic investment in the city's history. The funds will be disbursed over 18 months beginning April 2023"),
 Document(metadata={'article_id': 'ART-001', 'title': 'City Council Approves ₹500 Crore Infrastructure Plan', 'date': '2023-03-15', 'author': 'Priya Menon', 'section': 'City', 'chunk_index': 1, 'total_chunks': 3}, page_content='. The funds will be disbursed over 18 months beginning April 2023. Councillor Anita Rajan of Ward 7 raised concerns about contractor transparency, requesting monthly public audits. The plan includes ₹120

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19856.29it/s]


In [30]:
import chromadb
from langchain_community.vectorstores import Chroma
import os

os.makedirs(CHROMA_DB_PATH, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DB_PATH
)

vectorstore.persist()

In [31]:
# build retriever
base_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [32]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)

hf_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.3,
    do_sample=True
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

# advanced rag HyDe for retrieval
def generate_hypothetical_document(question: str) -> str:
    hyde_prompt = f"""You are a newspaper editor. Write a short, factual newspaper excerpt that would answer this question.
Question: {question}
Newspaper excerpt:"""

    hypothetical_doc = llm.invoke(hyde_prompt)
    return hypothetical_doc.strip()

test_question = "What environmental problems did Chennai face?"
hyp_doc = generate_hypothetical_document(test_question)

print(f"HyDE Test: ")
print(f"   Original Question : {test_question}")
print(f"   Hypothetical Doc  : {hyp_doc}")

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 6607.85it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', '

HyDE Test: 
   Original Question : What environmental problems did Chennai face?
   Hypothetical Doc  : You are a newspaper editor. Write a short, factual newspaper excerpt that would answer this question.
Question: What environmental problems did Chennai face?
Newspaper excerpt:


In [33]:
from langchain_core.documents import Document
from langchain_core.retrievers import  BaseRetriever
from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
from typing import List
from pydantic import Field

class HyDERetriever(BaseRetriever):
    vectorstore: object = Field(description="ChromaDB vector store")
    llm: object = Field(description="LLM for hypothetical doc generation")
    embedding_model: object = Field(description="Embedding model")
    k: int = Field(default=5, description="Number of docs to retrieve")

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:

        hyde_prompt = f"""Write a short factual newspaper excerpt answering this question.
Question: {query}
Excerpt:"""
        hypothetical_doc = self.llm.invoke(hyde_prompt).strip()
        hyp_embedding = self.embedding_model.embed_query(hypothetical_doc)
        results = self.vectorstore.similarity_search_by_vector(
            hyp_embedding,
            k=self.k
        )

        return results


hyde_retriever = HyDERetriever(
    vectorstore=vectorstore,
    llm=llm,
    embedding_model=embedding_model,
    k=5
)

In [34]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

ANSWER_PROMPT_TEMPLATE = """You are the Archive Desk — an editorial research assistant for a regional newspaper.

RULES:
1. Answer ONLY using the context passages below.
2. If the context does not contain enough information, say: "UNSUPPORTED: The archive does not contain sufficient information to answer this question."
3. Always mention the article title and date when you cite information.
4. Do NOT invent facts, statistics, or quotes.

CONTEXT FROM ARCHIVE:
{context}

QUESTION: {question}

ANSWER (based only on above context):"""

answer_prompt = PromptTemplate(
    template=ANSWER_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

def format_context(docs: List[Document]) -> str:
    context_parts = []
    seen_articles = set()

    for doc in docs:
        meta = doc.metadata
        article_key = meta['article_id']

        if article_key not in seen_articles:
            seen_articles.add(article_key)
            header = f"[ARTICLE: {meta['title']} | {meta['date']} | By {meta['author']} | Section: {meta['section']}]"
        else:
            header = f"[CONTINUED: {meta['title']}]"

        context_parts.append(f"{header}\n{doc.page_content}")

    return "\n\n---\n\n".join(context_parts)


def check_relevance(docs: List[Document], question: str, threshold: float = 0.25) -> bool:
    if not docs:
        return False

    question_words = set(question.lower().split())
    stop_words = {'what', 'when', 'where', 'who', 'how', 'did', 'was', 'is',
                  'are', 'the', 'a', 'an', 'in', 'of', 'to', 'for', 'and'}
    question_keywords = question_words - stop_words

    combined_text = " ".join([d.page_content.lower() for d in docs[:3]])
    overlap = sum(1 for kw in question_keywords if kw in combined_text)

    coverage = overlap / max(len(question_keywords), 1)
    return coverage >= threshold

In [35]:
from langsmith import traceable

@traceable(name="archive_desk_rag")
def ask_archive(question: str, use_hyde: bool = True) -> dict:
    if use_hyde:
        retrieved_docs = hyde_retriever.invoke(question)
    else:
        retrieved_docs = base_retriever.invoke(question)

    is_relevant = check_relevance(retrieved_docs, question)

    if not is_relevant:
        blocked_response = {
            "answer": "UNSUPPORTED: The archive does not contain sufficient information to answer this question. The available articles do not cover this topic.",
            "sources": [],
            "is_supported": False,
            "retrieved_chunks": retrieved_docs
        }
        return blocked_response

    context_str = format_context(retrieved_docs)

    formatted_prompt = answer_prompt.format(
        context=context_str,
        question=question
    )

    raw_answer = llm.invoke(formatted_prompt)

    sources = []
    seen = set()
    for doc in retrieved_docs:
        meta = doc.metadata
        key = meta['article_id']
        if key not in seen:
            seen.add(key)
            sources.append({
                "id": meta['article_id'],
                "title": meta['title'],
                "date": meta['date'],
                "author": meta['author'],
                "section": meta['section']
            })

    result = {
        "answer": raw_answer.strip(),
        "sources": sources,
        "is_supported": True,
        "retrieved_chunks": retrieved_docs
    }

    for src in sources:
        print(f"   [{src['id']}] {src['title']} | {src['date']} | By {src['author']}")

    return result

In [36]:
r1 = ask_archive("What was the Chennai City Council's infrastructure investment plan?")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (611 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [ART-001] City Council Approves ₹500 Crore Infrastructure Plan | 2023-03-15 | By Priya Menon


In [37]:
r2 = ask_archive("Tell me about EcoRide and their electric vehicle startup.")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [ART-002] Local Startup EcoRide Raises ₹40 Crore in Series A | 2023-06-02 | By Karthik Sundaram


In [38]:
r3 = ask_archive("Who won the FIFA World Cup in 2022?")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [39]:
r4 = ask_archive("What health and environmental challenges were reported?")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [40]:
r5 = ask_archive("What achievements were reported in Tamil Nadu education?")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [ART-004] Government School Achieves 100% Class X Pass Rate | 2023-05-10 | By Meena Krishnaswamy


In [19]:
from langsmith import Client

ls_client = Client()

eval_examples = [
    {
        "question": "How much did the Chennai City Council allocate for infrastructure?",
        "expected_answer": "The Chennai City Council approved a ₹500 crore infrastructure plan.",
        "expected_sources": ["ART-001"]
    },
    {
        "question": "Who founded EcoRide and how much did they raise?",
        "expected_answer": "EcoRide was founded by Deepa Nair and Vikram Anand. They raised ₹40 crore in Series A funding.",
        "expected_sources": ["ART-002"]
    },
    {
        "question": "How many families were displaced in the Tambaram floods?",
        "expected_answer": "Approximately 1,200 families were displaced.",
        "expected_sources": ["ART-005"]
    },
    {
        "question": "What world record did Arun Kumar break?",
        "expected_answer": "Arun Kumar set a championship record at the National Cycling Championship, finishing the 100km road race in 2 hours 18 minutes.",
        "expected_sources": ["ART-006"]
    },
    {
        "question": "Who is the Prime Minister of Japan?",  # Unsupported
        "expected_answer": "UNSUPPORTED",
        "expected_sources": []
    },
    {
        "question": "What is the status of AIIMS Chennai's paediatric ward?",
        "expected_answer": "AIIMS Chennai plans to open a 60-bed paediatric oncology ward by March 2024.",
        "expected_sources": ["ART-008"]
    },
    {
        "question": "What did Anna University researchers develop?",
        "expected_answer": "Anna University developed a low-cost water purification device for ₹1,200 per unit.",
        "expected_sources": ["ART-013"]
    },
    {
        "question": "What was the plastic waste collected at Marina Beach?",
        "expected_answer": "Over 8 tonnes of plastic waste was collected in a three-day drive at Marina Beach.",
        "expected_sources": ["ART-003"]
    }
]

print(f"Evaluation dataset: {len(eval_examples)} test cases")
for i, ex in enumerate(eval_examples, 1):
    label = "Supported" if ex['expected_sources'] else "Should be blocked"
    print(f"   [{i}] {label}: {ex['question'][:60]}")

Evaluation dataset: 8 test cases
   [1] Supported: How much did the Chennai City Council allocate for infrastru
   [2] Supported: Who founded EcoRide and how much did they raise?
   [3] Supported: How many families were displaced in the Tambaram floods?
   [4] Supported: What world record did Arun Kumar break?
   [5] Should be blocked: Who is the Prime Minister of Japan?
   [6] Supported: What is the status of AIIMS Chennai's paediatric ward?
   [7] Supported: What did Anna University researchers develop?
   [8] Supported: What was the plastic waste collected at Marina Beach?


In [20]:
DATASET_NAME = "archive-desk-eval-v1"

# delete existing dataset if it exists
try:
    existing = ls_client.read_dataset(dataset_name=DATASET_NAME)
    ls_client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset: {DATASET_NAME}")
except Exception:
    pass

# Create new dataset
dataset = ls_client.create_dataset(
    dataset_name=DATASET_NAME,
    description="Evaluation set for The Archive Desk RAG system"
)

# Upload examples
ls_client.create_examples(
    inputs=[{"question": ex["question"]} for ex in eval_examples],
    outputs=[{"expected_answer": ex["expected_answer"], "expected_sources": ex["expected_sources"]}
             for ex in eval_examples],
    dataset_id=dataset.id
)

print(f"Dataset '{DATASET_NAME}' uploaded to LangSmith with {len(eval_examples)} examples")

Deleted existing dataset: archive-desk-eval-v1
Dataset 'archive-desk-eval-v1' uploaded to LangSmith with 8 examples


In [21]:
from langsmith.evaluation import evaluate, StringEvaluator

def rag_target(inputs: dict) -> dict:
    result = ask_archive(inputs["question"])
    return {
        "output": result["answer"],
        "sources": [s["id"] for s in result["sources"]],
        "is_supported": result["is_supported"]
    }

from langsmith.evaluation import EvaluationResult

def blocking_accuracy_evaluator(run, example) -> EvaluationResult:
    expected = example.outputs["expected_answer"]
    actual = run.outputs.get("output", "")

    should_be_blocked = expected == "UNSUPPORTED"
    was_blocked = "UNSUPPORTED" in actual.upper()

    if should_be_blocked and was_blocked:
        score = 1.0
        comment = "Correctly refused to answer unsupported question"
    elif should_be_blocked and not was_blocked:
        score = 0.0
        comment = "FAILED: Should have blocked this question!"
    elif not should_be_blocked and was_blocked:
        score = 0.0
        comment = "FAILED: Blocked a question that was answerable"
    else:
        score = 1.0
        comment = "Correctly attempted to answer supported question"

    return EvaluationResult(key="blocking_accuracy", score=score, comment=comment)

def source_citation_evaluator(run, example) -> EvaluationResult:
    expected_sources = set(example.outputs["expected_sources"])
    actual_sources = set(run.outputs.get("sources", []))

    if not expected_sources:  # Unsupported question
        score = 1.0 if not actual_sources else 0.5
        return EvaluationResult(key="source_citation", score=score, comment="Unsupported question")

    correct = len(expected_sources & actual_sources)
    score = correct / len(expected_sources)
    comment = f"Found {correct}/{len(expected_sources)} expected sources"

    return EvaluationResult(key="source_citation", score=score, comment=comment)

print("Evaluators defined. Running evaluation...")
print("   This will call ask_archive() for all test cases. Takes a few minutes.")

Evaluators defined. Running evaluation...
   This will call ask_archive() for all test cases. Takes a few minutes.


In [22]:
from langsmith.evaluation import evaluate

eval_results = evaluate(
    rag_target,
    data=DATASET_NAME,
    evaluators=[
        blocking_accuracy_evaluator,
        source_citation_evaluator
    ],
    experiment_prefix="archive-desk-hyde",
    max_concurrency=1
)

print("\nEvaluation complete! View results at: https://smith.langchain.com")

View the evaluation results for experiment: 'archive-desk-hyde-8f915803' at:
https://smith.langchain.com/o/927c256e-eb3b-4aec-bf93-3c7f69c6f266/datasets/29337dc8-b8d1-40cb-98de-a6d19492526b/compare?selectedSessions=8897b7a0-303a-4adc-bdeb-44f4ecc6ca28




0it [00:00, ?it/s][transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
1it [00:00,  1.22it/s][transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2it [00:01,  1.17it/s][transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to th

   [ART-003] Marina Beach Plastic Drive Collects 8 Tonnes of Waste | 2022-09-25 | By Lakshmi Iyer


3it [00:02,  1.37it/s][transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
4it [00:03,  1.24it/s][transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [ART-008] AIIMS Chennai to Open Paediatric Cancer Ward in 2024 | 2023-09-14 | By Dr. Nalini Bose


...

   [ART-001] City Council Approves ₹500 Crore Infrastructure Plan | 2023-03-15 | By Priya Menon


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
8it [00:05,  1.41it/s]

   [ART-005] Flooding in Tambaram Displaces 1,200 Families | 2023-11-18 | By Arjun Balaji


8it [00:06,  1.26it/s]


Evaluation complete! View results at: https://smith.langchain.com
